# NeMo Agent Toolkit with the Movie MCP Server

Connect the movie database MCP server from Notebook 3 to a Nemotron-powered ReAct agent, ask natural-language movie questions, and inspect workflow traces with Arize Phoenix.

Run `03_movie_database_mcp.ipynb` first. It creates `movie_db.py` and `movie_server.py`, which this notebook launches and connects to.

## Setup

This notebook accepts either `NVIDIA_API_KEY` or `NGC_API_KEY` from the environment. Credentials are never written to the generated YAML files.

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("NAT_TELEMETRY_ENABLED", "false")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "movie.sqlite").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

required_files = [
    Path("data/movie.sqlite"),
    Path("movie_db.py"),
    Path("movie_server.py"),
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Run Notebook 3 first. Missing: " + ", ".join(missing_files)
    )

if not os.environ.get("NVIDIA_API_KEY"):
    if os.environ.get("NGC_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = os.environ["NGC_API_KEY"]
    else:
        raise EnvironmentError("Set NVIDIA_API_KEY or NGC_API_KEY before continuing")

print(f"Project root: {PROJECT_ROOT}")
print("NVIDIA API credential: available")

## Start the Movie MCP Server

Launch the high-level FastMCP server generated by Notebook 3. If `MCP_PORT` is not set, an available local port is selected.

In [ ]:
import socket
import subprocess
import sys
import time


def wait_for_port(process, host: str, port: int, log_path: Path, timeout: int = 30):
    """Wait for a subprocess to bind a TCP port or raise with its logs."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            details = log_path.read_text() if log_path.exists() else "No logs found"
            raise RuntimeError(details)
        try:
            with socket.create_connection((host, port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.25)
    process.terminate()
    raise TimeoutError(f"Process did not bind to {host}:{port}")


if "MCP_PORT" in os.environ:
    MCP_PORT = int(os.environ["MCP_PORT"])
else:
    with socket.socket() as port_socket:
        port_socket.bind(("127.0.0.1", 0))
        MCP_PORT = port_socket.getsockname()[1]
    os.environ["MCP_PORT"] = str(MCP_PORT)

mcp_log_path = Path("mcp_server.log")
mcp_log = mcp_log_path.open("w")
mcp_server_process = subprocess.Popen(
    [sys.executable, "movie_server.py", "data/movie.sqlite"],
    stdout=mcp_log,
    stderr=subprocess.STDOUT,
)
wait_for_port(mcp_server_process, "127.0.0.1", MCP_PORT, mcp_log_path)

MCP_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"
print(f"MCP server started on {MCP_URL} (PID: {mcp_server_process.pid})")
print("Logs: mcp_server.log")

## Define a NAT Workflow

NeMo Agent Toolkit workflows are defined in YAML. This workflow has three main sections:

- `function_groups`: the external movie MCP server;
- `llms`: the NVIDIA NIM-hosted Nemotron model; and
- `workflow`: a ReAct agent that can call the movie tools.

`parse_agent_response_max_retries: 3` allows the agent to recover when an LLM response does not follow the ReAct format.

In [ ]:
%%writefile movie_workflow.yml
function_groups:
  mcp_movies:
    _type: mcp_client
    server:
      transport: streamable-http
      url: "http://127.0.0.1:${MCP_PORT}/mcp"

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: https://integrate.api.nvidia.com/v1
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.0
    max_tokens: 1024

workflow:
  _type: react_agent
  tool_names:
    - mcp_movies
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3


Run the workflow with a simple movie query. The agent should call `search_movies` and summarize the returned rows.

In [ ]:
!nat run --config_file movie_workflow.yml --input "movies rated over 8.5"

## Start the Phoenix Observability Server

Arize Phoenix captures traces from the NeMo Agent Toolkit workflow so you can inspect LLM calls, tool invocations, latency, token usage, and errors. The bootcamp environment forwards port 6006 to the same port on your local machine.

In [ ]:
import shutil

PHOENIX_PORT = int(os.environ.get("PHOENIX_PORT", "6006"))
os.environ["PHOENIX_PORT"] = str(PHOENIX_PORT)

phoenix_executable = shutil.which("phoenix")
if phoenix_executable is None:
    raise FileNotFoundError(
        "The phoenix CLI is missing. Run `uv sync` from the repository root."
    )

phoenix_log_path = Path("phoenix.log")
phoenix_log = phoenix_log_path.open("w")
phoenix_process = subprocess.Popen(
    [phoenix_executable, "serve"],
    stdout=phoenix_log,
    stderr=subprocess.STDOUT,
)
wait_for_port(phoenix_process, "127.0.0.1", PHOENIX_PORT, phoenix_log_path, 60)

print(
    f"Phoenix started on http://127.0.0.1:{PHOENIX_PORT} "
    f"(PID: {phoenix_process.pid})"
)
print("Open http://localhost:6006 through the bootcamp port forwarding")
print("Logs: phoenix.log")

### Reading Traces

After the traced workflow runs, open the `movie-mcp` project in Phoenix. The span tree shows the ReAct workflow, NIM calls, the `search_movies` tool invocation, and the MCP request. Useful debugging patterns include:

- **Wrong answer:** locate the first span whose output contains the mistake.
- **Slow response:** compare span durations and look for retries.
- **Tool errors:** filter spans by error status and inspect tool arguments.
- **Token growth:** compare token totals across repeated runs.

## Configure Phoenix Tracing

Telemetry is configured in YAML. Every function, LLM, and tool in the workflow inherits the Phoenix exporter without explicit instrumentation code.

In [ ]:
%%writefile movie_workflow_tracing.yml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://127.0.0.1:${PHOENIX_PORT}/v1/traces
        project: movie-mcp

function_groups:
  mcp_movies:
    _type: mcp_client
    server:
      transport: streamable-http
      url: "http://127.0.0.1:${MCP_PORT}/mcp"

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-nano-30b-a3b
    base_url: https://integrate.api.nvidia.com/v1
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.0
    max_tokens: 1024

workflow:
  _type: react_agent
  tool_names:
    - mcp_movies
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3


Run a specific query and then inspect its trace in Phoenix:

In [ ]:
!nat run --config_file movie_workflow_tracing.yml --input "what is the rating of the movie The Dark Knight Rises?"

### Review the Trace

Open `http://localhost:6006`, select the `movie-mcp` project, and inspect the latest trace. A successful run contains at least one NIM span and one `search_movies` MCP tool span. The tool span should show the title filter and the JSON rows returned from SQLite.

## Optional Challenge: Cover All Tables

The current tool only searches six columns in the `IMDB` table. Extend `MovieDB` and the FastMCP server to answer questions that need the `earning` and `genre` tables or additional IMDB columns.

| Question | What it exercises |
| --- | --- |
| What's the worldwide gross of Inception? | `IMDB` and `earning` |
| Which biography has the highest IMDB rating? | `IMDB` and `genre` |
| What's the longest movie rated 8.5 or higher? | `Runtime` and `Rating` |
| Which movie has the biggest rating gap between male and female viewers? | `VotesM` and `VotesF` |
| Which movie is most loved by viewers under 18? | `VotesU18` |
| Which movie has the biggest disagreement between critics and audiences? | `MetaCritic` and `Rating` |
| Which comedy had the best return on investment? | all three tables and arithmetic |

After adding tools, restart the MCP process, verify them with `nat mcp client tool list`, rerun a workflow question, and confirm the new tool spans in Phoenix.

## Cleanup

Stop both background processes when you finish.

In [ ]:
for name, process, process_log in [
    ("MCP server", mcp_server_process, mcp_log),
    ("Phoenix", phoenix_process, phoenix_log),
]:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    finally:
        process_log.close()
    print(f"{name} (PID: {process.pid}) stopped")

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.